# AI Operator Project — Starter Notebook

This notebook trains the baseline model, generates SHAP and LIME explanations, and saves the model for the dashboard.

### Steps:
1. Load dataset
2. Train/test split
3. Train XGBoost model with progress bar
4. Evaluate model
5. Generate SHAP values + plots
6. Generate LIME explanation
7. Save model for dashboard


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
from xgboost import XGBClassifier
import shap
from lime.lime_tabular import LimeTabularExplainer
from tqdm import tqdm
import joblib

print("Libraries loaded.")

## Load Dataset

In [ ]:
df = pd.read_csv("../data/ai4i2020.csv")
df.head()

## Train/Test Split

In [ ]:
X = df.drop("Machine failure", axis=1)
y = df["Machine failure"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train/Test split complete.")

## Train Model (with progress bar)

In [ ]:
model = XGBClassifier(n_estimators=200, learning_rate=0.05)

for _ in tqdm(range(1), desc="Training model"):
    model.fit(X_train, y_train)

print("Model training complete.")

## Evaluate Model

In [ ]:
preds = model.predict(X_test)
probs = model.predict_proba(X_test)[:,1]

print("Accuracy:", accuracy_score(y_test, preds))
print("ROC-AUC:", roc_auc_score(y_test, probs))

## SHAP Explanations

In [ ]:
shap.initjs()
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

print("SHAP values generated.")

In [ ]:
shap.summary_plot(shap_values, X_test)

## LIME Explanation (Single Instance)

In [ ]:
lime_explainer = LimeTabularExplainer(
    training_data=np.array(X_train),
    feature_names=X.columns,
    class_names=["No Failure", "Failure"],
    mode="classification"
)

i = 0  # first test sample
lime_exp = lime_explainer.explain_instance(
    data_row=X_test.iloc[i],
    predict_fn=model.predict_proba
)

lime_exp.show_in_notebook()

## Save Model for Dashboard

In [ ]:
joblib.dump(model, "../models/model.pkl")
print("Model saved to ../models/model.pkl")

# Notebook Ready for Dashboard

Next step: Build the Streamlit dashboard that loads `model.pkl`, SHAP values, and LIME explanations.